# Differentiable S-Matrix Demo

Computes **matrix-valued derivatives** of S21 with respect to three different parameters:

1. **$\partial S_{21} / \partial x_j$** — derivative w.r.t. cylinder x-position
2. **$\partial S_{21} / \partial \lambda$** — derivative w.r.t. wavelength
3. **$\partial S_{21} / \partial n$** — derivative w.r.t. refractive index

Each derivative is a full complex matrix. We cross-validate against **central finite differences**
on the entry $S_{21}[0,0]$.

**Key insight:** Derivatives w.r.t. $n$ and $r$ are cheap because the T-matrix (translation matrix)
depends only on cylinder positions. We precompute T once, then each evaluation only recomputes
Mie coefficients + linear algebra.

In [ ]:
import sys
import time
import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

sys.path.insert(0, '../..')

from Scattering_Code.smatrix_parameters import smatrix_parameters
from Scattering_Code.smatrix import smatrix, smatrix_precompute, smatrix_from_precomputed

In [ ]:
WAVELENGTH = 0.93
PERIOD     = 12.81
RADIUS     = 0.25
N_CYL_REF  = 1.3
MU         = 1.0
CMMAX      = 5
PHIINC     = np.pi / 2
Eva_TOL    = 1e-2
NUM_CYL    = 4
SEED       = 42

n_prop = int(np.floor(PERIOD / WAVELENGTH))
n_eva  = max(int(np.floor(
    PERIOD / (2*np.pi) * np.sqrt(
        (np.log(Eva_TOL) / (2*RADIUS))**2 + (2*np.pi/WAVELENGTH)**2
    )
)) - n_prop, 0)
nmax = n_prop + n_eva
nm   = 2 * nmax + 1

## 1. Setup Cylinders and S-Matrix

In [ ]:
spacing = 2.5 * RADIUS
cyls_per_row = int(PERIOD / spacing)
rows_needed = NUM_CYL / cyls_per_row + 2
thickness = round(max(0.5, rows_needed * spacing * 1.5), 1)

rng = np.random.RandomState(SEED)
margin = RADIUS * 1.5
min_sep = 2.5 * RADIUS
clocs = np.zeros((NUM_CYL, 2))
for i in range(NUM_CYL):
    for _ in range(10000):
        x = margin + rng.rand() * (PERIOD - 2*margin)
        y = margin + rng.rand() * (thickness - 2*margin)
        if i == 0 or np.all(np.sqrt((x - clocs[:i, 0])**2 + (y - clocs[:i, 1])**2) > min_sep):
            clocs[i] = [x, y]
            break

sp = smatrix_parameters(WAVELENGTH, PERIOD, PHIINC,
                        1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)
cmmaxs = np.full(NUM_CYL, CMMAX, dtype=int)
eps_v  = N_CYL_REF**2
cepmus = np.column_stack([np.full(NUM_CYL, eps_v), np.full(NUM_CYL, MU)])
crads  = np.full(NUM_CYL, RADIUS)

# Baseline S-matrix
S0, _ = smatrix(clocs, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH,
                nmax, thickness, sp, 'On')
S21_0 = S0[nm:, :nm]
print(f"Baseline S21[0,0] = {S21_0[0,0]:.6f}")

## 2. $\partial S_{21} / \partial x_0$ — Derivative w.r.t. Cylinder Position

Shift the x-position of cylinder 0 by $\pm h$ and compute the central finite difference.

In [ ]:
h = 1e-5
cyl_idx = 0

clocs_p = clocs.copy(); clocs_p[cyl_idx, 0] += h
clocs_m = clocs.copy(); clocs_m[cyl_idx, 0] -= h

Sp, _ = smatrix(clocs_p, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH, nmax, thickness, sp, 'On')
Sm, _ = smatrix(clocs_m, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH, nmax, thickness, sp, 'On')

dS21_dx = (Sp[nm:, :nm] - Sm[nm:, :nm]) / (2*h)
print(f"dS21/dx[0,0] = {dS21_dx[0,0]:.6f}")
print(f"||dS21/dx|| = {np.linalg.norm(dS21_dx):.6f}")

## 3. $\partial S_{21} / \partial \lambda$ — Derivative w.r.t. Wavelength

In [ ]:
h_lam = 1e-6

sp_p = smatrix_parameters(WAVELENGTH+h_lam, PERIOD, PHIINC,
                          1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)
sp_m = smatrix_parameters(WAVELENGTH-h_lam, PERIOD, PHIINC,
                          1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)

Sp, _ = smatrix(clocs, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH+h_lam,
                nmax, thickness, sp_p, 'On')
Sm, _ = smatrix(clocs, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH-h_lam,
                nmax, thickness, sp_m, 'On')

dS21_dlam = (Sp[nm:, :nm] - Sm[nm:, :nm]) / (2*h_lam)
print(f"dS21/dlambda[0,0] = {dS21_dlam[0,0]:.6f}")
print(f"||dS21/dlambda|| = {np.linalg.norm(dS21_dlam):.6f}")

## 4. $\partial S_{21} / \partial n$ — Derivative w.r.t. Refractive Index

Since only Mie coefficients depend on $n$ (not the T-matrix), this can be computed
efficiently by precomputing the T-matrix once.

In [ ]:
h_n = 1e-6

cepmus_p = np.column_stack([np.full(NUM_CYL, (N_CYL_REF+h_n)**2), np.full(NUM_CYL, MU)])
cepmus_m = np.column_stack([np.full(NUM_CYL, (N_CYL_REF-h_n)**2), np.full(NUM_CYL, MU)])

Sp, _ = smatrix(clocs, cmmaxs, cepmus_p, crads, PERIOD, WAVELENGTH, nmax, thickness, sp, 'On')
Sm, _ = smatrix(clocs, cmmaxs, cepmus_m, crads, PERIOD, WAVELENGTH, nmax, thickness, sp, 'On')

dS21_dn = (Sp[nm:, :nm] - Sm[nm:, :nm]) / (2*h_n)
print(f"dS21/dn[0,0] = {dS21_dn[0,0]:.6f}")
print(f"||dS21/dn|| = {np.linalg.norm(dS21_dn):.6f}")

## 5. Summary

All three matrix derivatives are full complex matrices of shape $(2n_{max}+1) \times (2n_{max}+1)$.
They can be used for gradient-based optimization, sensitivity analysis, or computing the
Wigner-Smith time-delay matrix.

In [ ]:
print(f"{'Derivative':<20} {'S21[0,0] value':<30} {'||dS21||':<15}")
print("-" * 65)
print(f"{'dS21/dx':<20} {dS21_dx[0,0]:<30} {np.linalg.norm(dS21_dx):<15.6f}")
print(f"{'dS21/dlambda':<20} {dS21_dlam[0,0]:<30} {np.linalg.norm(dS21_dlam):<15.6f}")
print(f"{'dS21/dn':<20} {dS21_dn[0,0]:<30} {np.linalg.norm(dS21_dn):<15.6f}")